In [1]:
import requests
from pathlib import Path

DATASET_ID = "6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5"
CKAN_API = f"https://datos.produccion.gob.ar/api/3/action/package_show?id={DATASET_ID}"

# Consultar la API para obtener URLs actuales
response = requests.get(CKAN_API, timeout=30)
response.raise_for_status()
resources = response.json()["result"]["resources"]

# Construir mapa dia -> url desde los recursos que sean ZIPs de precios
url_por_dia = {
    r["name"].lower(): r["url"]
    for r in resources
    if r["url"].endswith(".zip") and "sepa_" in r["url"]
}

print("URLs disponibles:")
for dia, url in url_por_dia.items():
    print(f"  {dia}: {url}")

URLs disponibles:
  viernes: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/91bc072a-4726-44a1-85ec-4a8467aad27e/download/sepa_viernes.zip
  jueves: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/d076720f-a7f0-4af8-b1d6-1b99d5a90c14/download/sepa_jueves.zip
  miércoles: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/1e92cd42-4f94-4071-a165-62c4cb2ce23c/download/sepa_miercoles.zip
  martes: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/9dc06241-cc83-44f4-8e25-c9b1636b8bc8/download/sepa_martes.zip
  lunes: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/0a9069a9-06e8-4f98-874d-da5578693290/download/sepa_lunes.zip
  domingo: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/f8e75128-515a-436e-bf8d-5c63a62f2005/download/sepa_domingo.zip
  sábado: https://datos.pr

In [2]:
from datetime import date

DIAS_ES = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

dia_hoy = DIAS_ES[date.today().weekday()]
url = url_por_dia[dia_hoy]
dest = Path(f"../data/{dia_hoy}.zip")
dest.parent.mkdir(parents=True, exist_ok=True)

print(f"Descargando ZIP de {dia_hoy}...")
r = requests.get(url, stream=True, timeout=300)
r.raise_for_status()

dest.write_bytes(r.content)
print(f"Descargado: {dest} ({dest.stat().st_size / 1024 / 1024:.1f} MB)")

Descargando ZIP de martes...
Descargado: ../data/martes.zip (293.4 MB)


In [3]:
import zipfile

extract_dir = Path(f"../data/{dia_hoy}")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(dest) as z:
    print("Contenido del ZIP:")
    for name in z.namelist():
        info = z.getinfo(name)
        print(f"  {name} ({info.file_size / 1024 / 1024:.1f} MB)")
    z.extractall(extract_dir)

print(f"\nExtraído en: {extract_dir}")

Contenido del ZIP:
  2026-03-17/ (0.0 MB)
  2026-03-17/sepa_1_comercio-sepa-23_2026-03-17_09-05-11.zip (0.0 MB)
  2026-03-17/sepa_1_comercio-sepa-4_2026-03-17_09-05-11.zip (0.0 MB)
  2026-03-17/sepa_1_comercio-sepa-15_2026-03-17_09-05-11.zip (75.5 MB)
  2026-03-17/sepa_2_comercio-sepa-36_2026-03-17_01-05-08.zip (0.0 MB)
  2026-03-17/sepa_2_comercio-sepa-6_2026-03-17_01-05-08.zip (0.5 MB)
  2026-03-17/sepa_1_comercio-sepa-21_2026-03-17_09-05-11.zip (2.2 MB)
  2026-03-17/sepa_2_comercio-sepa-13_2026-03-17_01-05-08.zip (15.5 MB)
  2026-03-17/sepa_1_comercio-sepa-12_2026-03-17_09-05-11.zip (7.9 MB)
  2026-03-17/sepa_1_comercio-sepa-20_2026-03-17_09-05-11.zip (0.1 MB)
  2026-03-17/sepa_1_comercio-sepa-19_2026-03-17_09-05-11.zip (0.2 MB)
  2026-03-17/sepa_2_comercio-sepa-2_2026-03-17_01-05-08.zip (18.8 MB)
  2026-03-17/sepa_2_comercio-sepa-10_2026-03-17_01-05-08.zip (124.0 MB)
  2026-03-17/sepa_2_comercio-sepa-5_2026-03-17_01-05-08.zip (0.1 MB)
  2026-03-17/sepa_1_comercio-sepa-47_2026-03-17

In [4]:
import zipfile
from pathlib import Path

# Buscar ZIPs dentro de la carpeta extraída
inner_zips = list(extract_dir.rglob("*.zip"))
print(f"ZIPs encontrados: {len(inner_zips)}")
for z in inner_zips[:5]:
    print(f"  {z.name}")

# Extraer el primero y ver su contenido
sample_zip = inner_zips[0]
sample_dir = extract_dir / sample_zip.stem

with zipfile.ZipFile(sample_zip) as z:
    print(f"\nContenido de {sample_zip.name}:")
    for name in z.namelist():
        info = z.getinfo(name)
        print(f"  {name} ({info.file_size / 1024 / 1024:.1f} MB)")
    z.extractall(sample_dir)

print(f"\nExtraído en: {sample_dir}")

ZIPs encontrados: 18
  sepa_1_comercio-sepa-8_2026-03-17_09-05-11.zip
  sepa_1_comercio-sepa-15_2026-03-17_09-05-11.zip
  sepa_1_comercio-sepa-12_2026-03-17_09-05-11.zip
  sepa_1_comercio-sepa-3_2026-03-17_09-05-11.zip
  sepa_1_comercio-sepa-19_2026-03-17_09-05-11.zip

Contenido de sepa_1_comercio-sepa-8_2026-03-17_09-05-11.zip:
  comercio.csv (0.0 MB)
  productos.csv (4.9 MB)
  sucursales.csv (0.0 MB)

Extraído en: ../data/martes/sepa_1_comercio-sepa-8_2026-03-17_09-05-11


In [5]:
import pandas as pd

df_comercio   = pd.read_csv(sample_dir / "comercio.csv",   sep="|", encoding_errors="replace")
df_productos  = pd.read_csv(sample_dir / "productos.csv",  sep="|", encoding_errors="replace")
df_sucursales = pd.read_csv(sample_dir / "sucursales.csv", sep="|", encoding_errors="replace")

# print("=== comercio.csv ===")
# print(df_comercio.shape)
# display(df_comercio.head(3))

print("=== productos.csv ===")
print(df_productos.shape)
display(df_productos.head(100))

# print("=== sucursales.csv ===")
# print(df_sucursales.shape)
# display(df_sucursales.head(3))

=== productos.csv ===
(43273, 17)


/tmp/ipykernel_71305/206441945.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_productos  = pd.read_csv(sample_dir / "productos.csv",  sep="|", encoding_errors="replace")


,id_comercio,id_bandera,id_sucursal,id_producto,productos_ean,productos_descripcion,productos_cantidad_presentacion,productos_unidad_medida_presentacion,productos_marca,productos_precio_lista,productos_precio_referencia,productos_cantidad_referencia,productos_unidad_medida_referencia,productos_precio_unitario_promo1,productos_leyenda_promo1,productos_precio_unitario_promo2,productos_leyenda_promo2
0,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA PABILO DOBLE TRAMA 40x45,1.0,uni.,10 y 11 S.A,999.0,999.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
1,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA PABILO CLASICA 42x40cm,1.0,uni.,10 y 11 S.A,730.0,730.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
2,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA AMERICANA 42x38cm.,1.0,uni.,10 y 11 S.A,939.0,939.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
3,8,1.0,1.0,7.798118e+12,1.0,FRANELA LEYENDA CLASICA COLOR 40x40cm,1.0,uni.,10 y 11 S.A,1490.0,1490.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
4,8,1.0,1.0,7.798118e+12,1.0,TRAPO PISO LEYENDA GRIS.PREM.FRIZ. 50x60,1.0,uni.,10 y 11 S.A,2390.0,2390.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,8,1.0,1.0,7.790046e+12,1.0,GALLETITAS GRANIX SALVADO SIN SAL 230 GR,230.0,Gr,ASOC.ARG.ADV.7mo DIA (a.granix),1269.0,5517.39,0.23,Precio x 1 Kg,NaN,NaN,NaN,NaN
96,8,1.0,1.0,7.790046e+12,1.0,GALLETITAS GRANIX SALVADO 0 COLEST. 240,240.0,Gr,ASOC.ARG.ADV.7mo DIA (a.granix),1379.0,574.58,2.40,Precio x 100 Gr,NaN,NaN,NaN,NaN
97,8,1.0,1.0,7.790045e+12,1.0,GALLETITAS GRANAGUA SANDWICH 200gr,200.0,Gr,ASOC.ARG.ADV.7mo DIA (a.granix),1169.0,584.50,2.00,Precio x 100 Gr,NaN,NaN,NaN,NaN
98,8,1.0,1.0,7.790045e+12,1.0,GALLETITAS GRANIX LIGHT SIN SAL 360 GRS.,360.0,Gr,ASOC.ARG.ADV.7mo DIA (a.granix),2109.0,5858.33,0.36,Precio x 1 Kg,NaN,NaN,NaN,NaN


In [6]:
df_productos.dtypes

id_comercio                              object
id_bandera                              float64
id_sucursal                             float64
id_producto                             float64
productos_ean                           float64
productos_descripcion                    object
productos_cantidad_presentacion         float64
productos_unidad_medida_presentacion     object
productos_marca                          object
productos_precio_lista                  float64
productos_precio_referencia             float64
productos_cantidad_referencia           float64
productos_unidad_medida_referencia       object
productos_precio_unitario_promo1        float64
productos_leyenda_promo1                float64
productos_precio_unitario_promo2        float64
productos_leyenda_promo2                float64
dtype: object

In [7]:
print(df_productos.shape)
print(df_productos.dropna().shape)

selected_columns = ["id_comercio", "id_bandera", "id_sucursal", "id_producto", "productos_ean", "productos_descripcion", "productos_cantidad_presentacion", "productos_unidad_medida_presentacion", "productos_marca", "productos_precio_lista", "productos_precio_referencia", "productos_cantidad_referencia", "productos_unidad_medida_referencia"]
print(df_productos[selected_columns].dropna().shape)

filtered_df = df_productos[selected_columns].dropna()
filtered_df.head(5)

(43273, 17)
(0, 17)
(43110, 13)


,id_comercio,id_bandera,id_sucursal,id_producto,productos_ean,productos_descripcion,productos_cantidad_presentacion,productos_unidad_medida_presentacion,productos_marca,productos_precio_lista,productos_precio_referencia,productos_cantidad_referencia,productos_unidad_medida_referencia
0,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA PABILO DOBLE TRAMA 40x45,1.0,uni.,10 y 11 S.A,999.0,999.0,1.0,Precio x Unidad
1,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA PABILO CLASICA 42x40cm,1.0,uni.,10 y 11 S.A,730.0,730.0,1.0,Precio x Unidad
2,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA AMERICANA 42x38cm.,1.0,uni.,10 y 11 S.A,939.0,939.0,1.0,Precio x Unidad
3,8,1.0,1.0,7.798118e+12,1.0,FRANELA LEYENDA CLASICA COLOR 40x40cm,1.0,uni.,10 y 11 S.A,1490.0,1490.0,1.0,Precio x Unidad
4,8,1.0,1.0,7.798118e+12,1.0,TRAPO PISO LEYENDA GRIS.PREM.FRIZ. 50x60,1.0,uni.,10 y 11 S.A,2390.0,2390.0,1.0,Precio x Unidad


In [8]:
# productos con más de un precio distinto
precios_por_producto = filtered_df.groupby("id_producto")["productos_precio_lista"].nunique()
con_variacion = precios_por_producto[precios_por_producto > 1].sort_values(ascending=False)

print(f"Productos con precios distintos: {len(con_variacion)} de {precios_por_producto.shape[0]}")
display(con_variacion.head(10))

# ver detalle del que más variación tiene
id_ejemplo = con_variacion.index[0]
detalle = filtered_df[filtered_df["id_producto"] == id_ejemplo][["id_sucursal", "productos_descripcion", "productos_precio_lista"]]
print(f"\nDetalle id_producto={id_ejemplo:.0f}:")
display(detalle)

Productos con precios distintos: 26 de 4836


id_producto
2.003560e+12    2
2.203560e+12    2
2.403560e+12    2
7.450077e+12    2
7.622300e+12    2
7.790503e+12    2
7.791600e+12    2
7.791813e+12    2
7.791813e+12    2
7.791814e+12    2
Name: productos_precio_lista, dtype: int64


Detalle id_producto=2003560000000:


,id_sucursal,productos_descripcion,productos_precio_lista
1141,1.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
4540,2.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
8041,3.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
10520,4.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
14175,5.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
18407,6.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
22725,7.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
27433,8.0,QUESO CREMOSO SABORLAC EL KG.,8390.0
32095,9.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
36718,10.0,QUESO CREMOSO SABORLAC EL KG.,8990.0


In [9]:
# Cargar productos.csv de todos los ZIPs internos en un solo DataFrame
import pandas as pd
import zipfile
from pathlib import Path

COLS = [
    "id_comercio", "id_bandera", "id_sucursal", "id_producto",
    "productos_descripcion", "productos_marca",
    "productos_precio_lista",
]

frames = []
for zip_path in inner_zips:
    comercio_id = zip_path.stem.split("_")[2]  # ej: "comercio-sepa-8"
    with zipfile.ZipFile(zip_path) as z:
        if "productos.csv" not in z.namelist():
            continue
        with z.open("productos.csv") as f:
            df = pd.read_csv(f, sep="|", usecols=COLS, low_memory=False, encoding_errors="replace")
            df["source_zip"] = comercio_id
            frames.append(df)

all_df = pd.concat(frames, ignore_index=True)
all_df = all_df.dropna(subset=["id_producto", "productos_precio_lista"])
all_df["id_producto"] = all_df["id_producto"].astype("int64")

print(f"Filas totales:       {len(all_df):,}")
print(f"ZIPs cargados:       {len(frames)}")
print(f"Comercios únicos:    {all_df['id_comercio'].nunique()}")
print(f"Productos únicos:    {all_df['id_producto'].nunique()}")

Filas totales:       12,977,725
ZIPs cargados:       18
Comercios únicos:    17
Productos únicos:    82908


In [10]:
# Productos compartidos entre comercios y variación de precio entre ellos
productos_por_comercio = all_df.groupby("id_producto")["id_comercio"].nunique()
productos_multi_comercio = productos_por_comercio[productos_por_comercio > 1]

print(f"Productos en más de 1 comercio: {len(productos_multi_comercio):,} de {len(productos_por_comercio):,}")

# Para esos productos, calcular precio min/max por id_producto
variacion = (
    all_df[all_df["id_producto"].isin(productos_multi_comercio.index)]
    .groupby("id_producto")["productos_precio_lista"]
    .agg(precio_min="min", precio_max="max", n_comercios="count")
)
variacion["diferencia_pct"] = (variacion["precio_max"] - variacion["precio_min"]) / variacion["precio_min"] * 100
variacion = variacion[variacion["diferencia_pct"] > 0].sort_values("diferencia_pct", ascending=False)

print(f"Con diferencia de precio entre comercios: {len(variacion):,}")
display(variacion.head(15))

Productos en más de 1 comercio: 20,853 de 82,908
Con diferencia de precio entre comercios: 20,552


,precio_min,precio_max,n_comercios,diferencia_pct
id_producto,,,,
7790079010835,0.0,6215.0,72,inf
7790040669109,0.0,1450.0,161,inf
7790040720107,0.0,189.0,159,inf
7790040720206,0.0,1180.0,163,inf
7790040930506,0.0,162.0,159,inf
7790040932708,0.0,172.0,159,inf
7790895000430,0.0,6900.0,1591,inf
7622300840259,0.0,122.0,159,inf
7798128000691,0.0,4446.0,2231,inf


In [11]:
# Detalle del producto con mayor diferencia porcentual entre comercios
id_ejemplo = variacion.index[0]
detalle = (
    all_df[all_df["id_producto"] == id_ejemplo]
    [["id_comercio", "productos_descripcion", "productos_precio_lista"]]
    .drop_duplicates()
    .sort_values("productos_precio_lista")
)
print(f"id_producto: {id_ejemplo}")
display(detalle)

id_producto: 7790079010835


,id_comercio,productos_descripcion,productos_precio_lista
4963727,5,SALCHICHA PALADINI VIENA 12 u.,0.0
11319741,13,SALCH X12 PALADINI,5500.0
11412543,13,SALCH X12 PALADINI,5665.0
11554279,13,SALCH X12 PALADINI,5775.0
11402571,13,SALCH X12 PALADINI,5830.0
11762154,13,SALCH X12 PALADINI,5885.0
11952242,13,SALCH X12 PALADINI,5995.0
11501843,13,SALCH X12 PALADINI,6050.0
11753726,13,SALCH X12 PALADINI,6160.0
11803262,13,SALCH X12 PALADINI,6215.0


## Trailer line — detección y descarte

In [12]:
# Verificar la línea trailer en los 3 tipos de CSV
# Todos los CSVs terminan con: línea vacía + "Última actualización: ..."

for fname in ["comercio.csv", "productos.csv", "sucursales.csv"]:
    with open(sample_dir / fname, encoding="utf-8-sig") as f:
        lines = f.readlines()
    last_nonempty = [l.rstrip() for l in lines if l.strip()][-1]
    print(f"{fname}:")
    print(f"  total líneas: {len(lines)}")
    print(f"  última línea no vacía: {repr(last_nonempty)}")
    tiene_trailer = last_nonempty.startswith("Última actualización")
    print(f"  tiene trailer: {tiene_trailer}")
    print()

comercio.csv:
  total líneas: 4
  última línea no vacía: 'Última actualización: 2026-03-16T16:30:36-03:00'
  tiene trailer: True

productos.csv:
  total líneas: 43275
  última línea no vacía: 'Última actualización: 2026-03-16T16:30:52-03:00'
  tiene trailer: True

sucursales.csv:
  total líneas: 14
  última línea no vacía: 'Última actualización: 2026-03-16T16:30:36-03:00'
  tiene trailer: True



In [13]:
# Leer productos.csv descartando el trailer y con dtype=str (como hará el pipeline)
import pandas as pd

def leer_csv_sepa(path):
    """Lee un CSV SEPA: sep pipe, encoding UTF-8 BOM, descarta trailer, todo como string."""
    with open(path, encoding="utf-8-sig") as f:
        lines = f.readlines()
    # Quitar trailing vacías y línea de trailer
    while lines and not lines[-1].strip():
        lines.pop()
    if lines and lines[-1].startswith("Última actualización"):
        lines.pop()
    from io import StringIO
    return pd.read_csv(StringIO("".join(lines)), sep="|", dtype=str, low_memory=False)

df_prod_clean = leer_csv_sepa(sample_dir / "productos.csv")
df_com_clean  = leer_csv_sepa(sample_dir / "comercio.csv")
df_suc_clean  = leer_csv_sepa(sample_dir / "sucursales.csv")

print(f"productos (sin trailer): {df_prod_clean.shape}")
print(f"comercio  (sin trailer): {df_com_clean.shape}")
print(f"sucursales(sin trailer): {df_suc_clean.shape}")

productos (sin trailer): (43272, 17)
comercio  (sin trailer): (1, 8)
sucursales(sin trailer): (11, 21)


## Verificación: todos los sub-ZIPs tienen los 3 CSVs

In [14]:
import zipfile

EXPECTED = {"comercio.csv", "productos.csv", "sucursales.csv"}
ok = 0
for zp in inner_zips:
    with zipfile.ZipFile(zp) as z:
        files = set(z.namelist())
        missing = EXPECTED - files
        if missing:
            print(f"FALTA en {zp.name}: {missing}")
        else:
            ok += 1

print(f"Todos los sub-ZIPs tienen los 3 CSVs: {ok}/{len(inner_zips)}")

Todos los sub-ZIPs tienen los 3 CSVs: 18/18


## Inspección de comercio.csv

In [15]:
print("=== comercio.csv — muestra ===")
display(df_com_clean)

print(f"\nColumnas ({len(df_com_clean.columns)}):", list(df_com_clean.columns))
print(f"\nCardinalidad (id_comercio, id_bandera):")
print(f"  filas: {len(df_com_clean)}")
print(f"  pares únicos: {df_com_clean[['id_comercio','id_bandera']].drop_duplicates().shape[0]}")
print(f"\nCUIT: {df_com_clean['comercio_cuit'].unique()}")
print(f"Razón social: {df_com_clean['comercio_razon_social'].unique()}")
print(f"Bandera nombre: {df_com_clean['comercio_bandera_nombre'].unique()}")
print(f"Versión SEPA: {df_com_clean['comercio_version_sepa'].unique()}")

=== comercio.csv — muestra ===


,id_comercio,id_bandera,comercio_cuit,comercio_razon_social,comercio_bandera_nombre,comercio_bandera_url,comercio_ultima_actualizacion,comercio_version_sepa
0,8,1,30616491780,Mariano S.A.,Mariano Max,http://www.mmax.com.ar,2026-03-16T16:30:36-03:00,1



Columnas (8): ['id_comercio', 'id_bandera', 'comercio_cuit', 'comercio_razon_social', 'comercio_bandera_nombre', 'comercio_bandera_url', 'comercio_ultima_actualizacion', 'comercio_version_sepa']

Cardinalidad (id_comercio, id_bandera):
  filas: 1
  pares únicos: 1

CUIT: ['30616491780']
Razón social: ['Mariano S.A.']
Bandera nombre: ['Mariano Max']
Versión SEPA: ['1']


## Inspección de sucursales.csv

In [16]:
print("=== sucursales.csv — muestra ===")
display(df_suc_clean.head(5))
print(f"\nColumnas ({len(df_suc_clean.columns)}):", list(df_suc_clean.columns))

print("\nTipos de sucursal:")
print(df_suc_clean["sucursales_tipo"].value_counts())

print("\nProvincias (código ISO 3166-2):")
print(df_suc_clean["sucursales_provincia"].value_counts())

lat_null = df_suc_clean["sucursales_latitud"].isna().sum()
lon_null = df_suc_clean["sucursales_longitud"].isna().sum()
print(f"\nSucursales sin latitud: {lat_null}")
print(f"Sucursales sin longitud: {lon_null}")

web = df_suc_clean[df_suc_clean["sucursales_tipo"].str.lower() == "web"]
print(f"Sucursales tipo Web: {len(web)}")
if len(web) > 0:
    print(f"  lat nulos en Web: {web['sucursales_latitud'].isna().sum()}")

=== sucursales.csv — muestra ===


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,...,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
0,8,1,1,Sucursal 1,Supermercado,Av. 11 de Septiembre,2765,-31.443295,-64.148264,NaN,...,5014,CORDOBA,AR-X,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00
1,8,1,2,Sucursal 2,Hipermercado,Av. Arturo Capdevila,1831,-31.379569,-64.147652,NaN,...,5012,CORDOBA,AR-X,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00
2,8,1,3,Sucursal 3,Supermercado,Carlos Gimenez,35,-31.365896,-64.16644,NaN,...,5012,CORDOBA,AR-X,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00
3,8,1,4,Sucursal 4,Supermercado,Italia,363,-30.978745,-64.092456,NaN,...,5220,jesus maria,AR-X,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00-16:30 a 21:00,08:00 a 13:00
4,8,1,5,Sucursal 5,Hipermercado,Av. Santa Ana,3551,-31.412738,-64.230768,NaN,...,5010,CORDOBA,AR-X,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00,08:00 a 22:00



Columnas (21): ['id_comercio', 'id_bandera', 'id_sucursal', 'sucursales_nombre', 'sucursales_tipo', 'sucursales_calle', 'sucursales_numero', 'sucursales_latitud', 'sucursales_longitud', 'sucursales_observaciones', 'sucursales_barrio', 'sucursales_codigo_postal', 'sucursales_localidad', 'sucursales_provincia', 'sucursales_lunes_horario_atencion', 'sucursales_martes_horario_atencion', 'sucursales_miercoles_horario_atencion', 'sucursales_jueves_horario_atencion', 'sucursales_viernes_horario_atencion', 'sucursales_sabado_horario_atencion', 'sucursales_domingo_horario_atencion']

Tipos de sucursal:
sucursales_tipo
Hipermercado    8
Supermercado    3
Name: count, dtype: int64

Provincias (código ISO 3166-2):
sucursales_provincia
AR-X    11
Name: count, dtype: int64

Sucursales sin latitud: 0
Sucursales sin longitud: 0
Sucursales tipo Web: 0


## Análisis detallado de productos.csv

In [17]:
print(f"Columnas ({len(df_prod_clean.columns)}):", list(df_prod_clean.columns))
assert len(df_prod_clean.columns) == 17, f"Esperadas 17 columnas, encontradas {len(df_prod_clean.columns)}"
print("✓ 17 columnas confirmadas")

# Formato de precios: punto decimal
muestra_precios = df_prod_clean["productos_precio_lista"].dropna().head(5).tolist()
print(f"\nFormato de precio lista (5 muestras): {muestra_precios}")
print("✓ Separador decimal es punto" if all("." in p or p.isdigit() for p in muestra_precios) else "✗ Separador decimal inesperado")

# Los 4 campos de precio
campos_precio = ["productos_precio_lista", "productos_precio_referencia",
                 "productos_precio_unitario_promo1", "productos_precio_unitario_promo2"]
print("\nCampos de precio:")
for col in campos_precio:
    n_nulos = df_prod_clean[col].isna().sum()
    n_total = len(df_prod_clean)
    print(f"  {col}: {n_nulos:,} nulos ({n_nulos/n_total*100:.1f}%)")

Columnas (17): ['id_comercio', 'id_bandera', 'id_sucursal', 'id_producto', 'productos_ean', 'productos_descripcion', 'productos_cantidad_presentacion', 'productos_unidad_medida_presentacion', 'productos_marca', 'productos_precio_lista', 'productos_precio_referencia', 'productos_cantidad_referencia', 'productos_unidad_medida_referencia', 'productos_precio_unitario_promo1', 'productos_leyenda_promo1', 'productos_precio_unitario_promo2', 'productos_leyenda_promo2']
✓ 17 columnas confirmadas

Formato de precio lista (5 muestras): ['999', '730', '939', '1490', '2390']
✓ Separador decimal es punto

Campos de precio:
  productos_precio_lista: 0 nulos (0.0%)
  productos_precio_referencia: 0 nulos (0.0%)
  productos_precio_unitario_promo1: 43,272 nulos (100.0%)
  productos_precio_unitario_promo2: 43,272 nulos (100.0%)


In [18]:
# Porcentaje con promo1 y promo2
n = len(df_prod_clean)
con_promo1 = df_prod_clean["productos_precio_unitario_promo1"].notna().sum()
con_promo2 = df_prod_clean["productos_precio_unitario_promo2"].notna().sum()
con_alguna = (df_prod_clean["productos_precio_unitario_promo1"].notna() |
              df_prod_clean["productos_precio_unitario_promo2"].notna()).sum()

print(f"Filas con promo1: {con_promo1:,} ({con_promo1/n*100:.1f}%)")
print(f"Filas con promo2: {con_promo2:,} ({con_promo2/n*100:.1f}%)")
print(f"Filas con alguna promo: {con_alguna:,} ({con_alguna/n*100:.1f}%)")

# Distribución productos_ean
print(f"\nDistribución productos_ean:")
print(df_prod_clean["productos_ean"].value_counts())
ean_pct = (df_prod_clean["productos_ean"] == "1").sum() / n * 100
print(f"  EAN estándar (1): {ean_pct:.1f}%  |  Código interno (0): {100-ean_pct:.1f}%")

# Nulos por columna
print("\n% nulos por columna:")
nulos = (df_prod_clean.isna().sum() / n * 100).round(1)
print(nulos.to_string())

Filas con promo1: 0 (0.0%)
Filas con promo2: 0 (0.0%)
Filas con alguna promo: 0 (0.0%)

Distribución productos_ean:
productos_ean
1    42201
0     1071
Name: count, dtype: int64
  EAN estándar (1): 97.5%  |  Código interno (0): 2.5%

% nulos por columna:
id_comercio                               0.0
id_bandera                                0.0
id_sucursal                               0.0
id_producto                               0.0
productos_ean                             0.0
productos_descripcion                     0.0
productos_cantidad_presentacion           0.0
productos_unidad_medida_presentacion      0.3
productos_marca                           0.0
productos_precio_lista                    0.0
productos_precio_referencia               0.0
productos_cantidad_referencia             0.0
productos_unidad_medida_referencia        0.4
productos_precio_unitario_promo1        100.0
productos_leyenda_promo1                100.0
productos_precio_unitario_promo2        100.0
productos

## Consolidar los 3 CSVs de todos los sub-ZIPs

In [19]:
import zipfile
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
from io import StringIO, TextIOWrapper
from pathlib import Path

COLS_PROD = [
    "id_comercio", "id_bandera", "id_sucursal", "id_producto",
    "productos_ean", "productos_descripcion", "productos_cantidad_presentacion",
    "productos_unidad_medida_presentacion", "productos_marca",
    "productos_precio_lista", "productos_precio_referencia",
    "productos_cantidad_referencia", "productos_unidad_medida_referencia",
    "productos_precio_unitario_promo1", "productos_leyenda_promo1",
    "productos_precio_unitario_promo2", "productos_leyenda_promo2",
]

parquet_dir = extract_dir / "_parquet"
parquet_dir.mkdir(exist_ok=True)

def read_sepa_csv_from_zip(zf, fname, usecols=None):
    with zf.open(fname) as raw:
        lines = TextIOWrapper(raw, encoding="utf-8-sig").readlines()
    while lines and not lines[-1].strip():
        lines.pop()
    if lines and lines[-1].startswith("Última actualización"):
        lines.pop()
    # Usar callable para tolerar columnas ausentes en algún ZIP
    usecols_fn = (lambda c: c in set(usecols)) if usecols is not None else None
    return pd.read_csv(StringIO("".join(lines)), sep="|", dtype=str, low_memory=False, usecols=usecols_fn)


frames_com, frames_suc = [], []

for zip_path in inner_zips:
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        if "comercio.csv" in names:
            frames_com.append(read_sepa_csv_from_zip(z, "comercio.csv"))
        if "sucursales.csv" in names:
            frames_suc.append(read_sepa_csv_from_zip(z, "sucursales.csv"))
        if "productos.csv" in names:
            df = read_sepa_csv_from_zip(z, "productos.csv", usecols=COLS_PROD)
            pq.write_table(
                pa.Table.from_pandas(df, preserve_index=False),
                parquet_dir / f"{zip_path.stem}.parquet",
            )

# Comercio y sucursales son pequeños → RAM está bien
all_comercios  = pd.concat(frames_com,  ignore_index=True)
all_sucursales = pd.concat(frames_suc,  ignore_index=True)

# Productos queda en disco; all_productos es una vista DuckDB (no ocupa RAM)
con = duckdb.connect()
con.execute(f"CREATE VIEW all_productos AS SELECT * FROM read_parquet('{parquet_dir}/*.parquet')")

print(f"Consolidado — comercios:  {len(all_comercios):,} filas")
print(f"Consolidado — sucursales: {len(all_sucursales):,} filas")
print(f"Consolidado — productos:  {con.execute('SELECT COUNT(*) FROM all_productos').fetchone()[0]:,} filas (DuckDB, en disco)")
print(f"\nSub-ZIPs procesados: {len(inner_zips)}")
print(f"Parquets escritos en: {parquet_dir}")

Consolidado — comercios:  40 filas
Consolidado — sucursales: 2,712 filas
Consolidado — productos:  12,977,733 filas (DuckDB, en disco)

Sub-ZIPs procesados: 18
Parquets escritos en: ../data/martes/_parquet


## Unicidad de claves compuestas

In [21]:
# Clave compuesta en comercio: (id_comercio, id_bandera)
pk_com = ["id_comercio", "id_bandera"]
dups_com = all_comercios.duplicated(subset=pk_com).sum()
print(f"comercio.csv — filas: {len(all_comercios)}, duplicados en {pk_com}: {dups_com}")

# Clave compuesta en sucursales: (id_comercio, id_bandera, id_sucursal)
pk_suc = ["id_comercio", "id_bandera", "id_sucursal"]
dups_suc = all_sucursales.duplicated(subset=pk_suc).sum()
print(f"sucursales.csv — filas: {len(all_sucursales)}, duplicados en {pk_suc}: {dups_suc}")

# Clave compuesta en productos: (id_comercio, id_bandera, id_sucursal, id_producto) — via DuckDB
total_prod, distinct_prod = con.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(DISTINCT (id_comercio, id_bandera, id_sucursal, id_producto)) AS distinct_keys
    FROM all_productos
    WHERE id_comercio IS NOT NULL
      AND id_bandera  IS NOT NULL
      AND id_sucursal IS NOT NULL
      AND id_producto IS NOT NULL
""").fetchone()
dups_prod = total_prod - distinct_prod
print(f"productos.csv — filas: {total_prod:,}, duplicados en ['id_comercio','id_bandera','id_sucursal','id_producto']: {dups_prod:,}")

comercio.csv — filas: 40, duplicados en ['id_comercio', 'id_bandera']: 1
sucursales.csv — filas: 2712, duplicados en ['id_comercio', 'id_bandera', 'id_sucursal']: 39
productos.csv — filas: 12,977,725, duplicados en ['id_comercio','id_bandera','id_sucursal','id_producto']: 63


## Integridad referencial entre los 3 archivos

In [22]:
# sucursales → comercio: cada (id_comercio, id_bandera) en sucursales debe estar en comercio
keys_com = set(all_comercios[["id_comercio","id_bandera"]].apply(tuple, axis=1))
keys_suc_parent = set(all_sucursales[["id_comercio","id_bandera"]].apply(tuple, axis=1))
huerfanos_suc = keys_suc_parent - keys_com
print(f"Sucursales con (id_comercio,id_bandera) sin match en comercio: {len(huerfanos_suc)}")

# productos → sucursales: via DuckDB
keys_suc_full = set(all_sucursales[["id_comercio","id_bandera","id_sucursal"]].apply(tuple, axis=1))
prod_fk_keys = con.execute("""
    SELECT DISTINCT id_comercio, id_bandera, id_sucursal
    FROM all_productos
    WHERE id_comercio IS NOT NULL AND id_bandera IS NOT NULL AND id_sucursal IS NOT NULL
""").fetchdf()
keys_prod_parent = set(prod_fk_keys.apply(tuple, axis=1))
huerfanos_prod = keys_prod_parent - keys_suc_full
print(f"Productos con FK sin match en sucursales: {len(huerfanos_prod)}")

Sucursales con (id_comercio,id_bandera) sin match en comercio: 31
Productos con FK sin match en sucursales: 3


## Cardinalidad total consolidada

In [23]:
total_prod, unique_prod = con.execute("""
    SELECT COUNT(*), COUNT(DISTINCT id_producto)
    FROM all_productos
    WHERE id_producto IS NOT NULL
""").fetchone()

print("=== Cardinalidad del día ===")
print(f"Comercios únicos:    {all_comercios['id_comercio'].nunique():>8,}")
print(f"Banderas únicas:     {all_comercios['comercio_bandera_nombre'].nunique():>8,}")
print(f"Sucursales únicas:   {all_sucursales['id_sucursal'].nunique():>8,}  (global, puede repetirse entre comercios)")
print(f"Sucursales (clave completa): {all_sucursales[['id_comercio','id_bandera','id_sucursal']].drop_duplicates().shape[0]:>8,}")
print(f"Productos únicos (id):       {unique_prod:>8,}")
print(f"Total filas productos:       {total_prod:>8,}")

=== Cardinalidad del día ===
Comercios únicos:          24
Banderas únicas:           28
Sucursales únicas:      1,626  (global, puede repetirse entre comercios)
Sucursales (clave completa):    2,673
Productos únicos (id):         82,908
Total filas productos:       12,977,725


## Verificación de coordenadas

In [24]:
import pandas as pd

# Convertir a float para análisis numérico
suc = all_sucursales.copy()
suc["lat"] = pd.to_numeric(suc["sucursales_latitud"], errors="coerce")
suc["lon"] = pd.to_numeric(suc["sucursales_longitud"], errors="coerce")

# Bounding box Argentina: lat [-55, -21], lon [-73, -53]
LAT_MIN, LAT_MAX = -55.0, -21.0
LON_MIN, LON_MAX = -73.0, -53.0

con_coords = suc["lat"].notna() & suc["lon"].notna()
fuera_bbox = con_coords & ~(
    suc["lat"].between(LAT_MIN, LAT_MAX) & suc["lon"].between(LON_MIN, LON_MAX)
)

print(f"Total sucursales:         {len(suc):,}")
print(f"Con coordenadas:          {con_coords.sum():,}")
print(f"Sin coordenadas (lat/lon nulo): {(~con_coords).sum():,}")
print(f"Fuera del bounding box ARG: {fuera_bbox.sum():,}")

if fuera_bbox.sum() > 0:
    display(suc[fuera_bbox][["id_sucursal", "sucursales_nombre", "sucursales_tipo", "lat", "lon", "sucursales_provincia"]])

# Sucursales Web sin coordenadas (esperado)
web = suc[suc["sucursales_tipo"].str.lower() == "web"]
print(f"\nSucursales tipo Web: {len(web)}")
print(f"  Sin coordenadas (esperado): {(~(web['lat'].notna() & web['lon'].notna())).sum()}")

Total sucursales:         2,712
Con coordenadas:          2,599
Sin coordenadas (lat/lon nulo): 113
Fuera del bounding box ARG: 1


,id_sucursal,sucursales_nombre,sucursales_tipo,lat,lon,sucursales_provincia
2001,854,Avenida Bartolomé Mitre 3125/3145,Autoservicio,-34.64742,58.794224,AR-B



Sucursales tipo Web: 2
  Sin coordenadas (esperado): 0


## Comparación esquema real vs spec (Resolución 678/2020)

In [26]:
# Columnas esperadas según spec
SPEC = {
    "comercio.csv": [
        "id_comercio", "id_bandera", "comercio_cuit", "comercio_razon_social",
        "comercio_bandera_nombre", "comercio_bandera_url",
        "comercio_ultima_actualizacion", "comercio_version_sepa"
    ],
    "sucursales.csv": [
        "id_comercio", "id_bandera", "id_sucursal", "sucursales_nombre",
        "sucursales_tipo", "sucursales_calle", "sucursales_numero",
        "sucursales_latitud", "sucursales_longitud", "sucursales_observaciones",
        "sucursales_barrio", "sucursales_codigo_postal", "sucursales_localidad",
        "sucursales_provincia", "sucursales_lunes_horario_atencion",
        "sucursales_martes_horario_atencion", "sucursales_miercoles_horario_atencion",
        "sucursales_jueves_horario_atencion", "sucursales_viernes_horario_atencion",
        "sucursales_sabado_horario_atencion", "sucursales_domingo_horario_atencion"
    ],
    "productos.csv": [
        "id_comercio", "id_bandera", "id_sucursal", "id_producto", "productos_ean",
        "productos_descripcion", "productos_cantidad_presentacion",
        "productos_unidad_medida_presentacion", "productos_marca",
        "productos_precio_lista", "productos_precio_referencia",
        "productos_cantidad_referencia", "productos_unidad_medida_referencia",
        "productos_precio_unitario_promo1", "productos_leyenda_promo1",
        "productos_precio_unitario_promo2", "productos_leyenda_promo2"
    ],
}

prod_cols = [col for col, *_ in con.execute("DESCRIBE all_productos").fetchall()]

REAL = {
    "comercio.csv": list(all_comercios.columns),
    "sucursales.csv": list(all_sucursales.columns),
    "productos.csv": prod_cols,
}

for fname in SPEC:
    spec_cols = SPEC[fname]
    real_cols = REAL[fname]
    extra = set(real_cols) - set(spec_cols)
    falta = set(spec_cols) - set(real_cols)
    orden_ok = real_cols == spec_cols
    print(f"=== {fname} ===")
    print(f"  Spec: {len(spec_cols)} cols | Real: {len(real_cols)} cols")
    print(f"  Columnas extra en real:  {extra or '—'}")
    print(f"  Columnas faltantes:      {falta or '—'}")
    print(f"  Orden idéntico al spec:  {orden_ok}")
    print()

=== comercio.csv ===
  Spec: 8 cols | Real: 9 cols
  Columnas extra en real:  {'comercio_version_sepa                                             '}
  Columnas faltantes:      —
  Orden idéntico al spec:  False

=== sucursales.csv ===
  Spec: 21 cols | Real: 22 cols
  Columnas extra en real:  {'sucursales_domingo_horario_atencion                                                                                                   '}
  Columnas faltantes:      —
  Orden idéntico al spec:  False

=== productos.csv ===
  Spec: 17 cols | Real: 17 cols
  Columnas extra en real:  —
  Columnas faltantes:      —
  Orden idéntico al spec:  True



## Hallazgos — resumen ejecutivo

| Aspecto | Resultado |
|---|---|
| Estructura ZIP | 1 ZIP principal → carpeta fecha → 18 sub-ZIPs por comercio, cada uno con 3 CSVs |
| Separador | `\|` pipe, confirmado |
| Encoding | UTF-8 con BOM (`utf-8-sig`) |
| Trailer | Todos los CSVs terminan con línea vacía + `Última actualización: ...` → debe descartarse antes de parsear |
| comercio.csv | 8 cols spec + **1 col extra con trailing whitespace** en nombre; ~2 filas por sub-ZIP; **1 duplicado de PK** `(id_comercio, id_bandera)` |
| sucursales.csv | 21 cols spec + **1 col extra con trailing whitespace** en nombre; **39 duplicados de PK** `(id_comercio, id_bandera, id_sucursal)` → requiere dedup en staging |
| sucursales_tipo | Valores observados: Hipermercado / Supermercado / Autoservicio / **Web** (2 sucursales). ⚠️ Otros comercios tienen localidades en lugar de tipo (ej: "San Salvador de Jujuy") |
| productos.csv | 17 columnas, esquema exacto al spec; precios con punto decimal; promo1/promo2 opcionales; **63 duplicados de PK** `(id_comercio, id_bandera, id_sucursal, id_producto)` |
| Integridad referencial | **31 sucursales** con `(id_comercio, id_bandera)` sin match en comercio; **3 productos** con FK sin match en sucursales |
| Coordenadas | 2,599 sucursales con coords; **113 sin lat/lon**; **1 fuera del bounding box ARG** (longitud positiva +58.79 en AR-B — dato corrupto) |
| Sucursales Web | 2 sucursales tipo Web, ambas **con** coordenadas cargadas (inesperado, posiblemente incorrecto) |
| Schema vs spec | productos.csv coincide exactamente; comercio.csv y sucursales.csv tienen **1 columna extra con nombre que incluye trailing whitespace** → normalizar nombres de columna al leer |

**Decisiones de pipeline derivadas de estos hallazgos:**
- Leer con `dtype=str` en staging (no inferir tipos)
- Descarte de trailer antes de parsear
- Decodificar con `utf-8-sig` para manejar el BOM
- **Limpiar nombres de columna** (`str.strip()`) al leer — hay trailing whitespace en al menos comercio y sucursales
- Deduplicar los 3 CSVs en staging antes de cargar a dimensiones (hay dups en los 3)
- Normalizar `sucursales_tipo`: rechazar o mapear valores que no sean Hipermercado/Supermercado/Autoservicio/Tradicional/Web
- Filtrar o marcar sucursales Web con coordenadas (2 casos — verificar si son válidas)
- Descartar / marcar la sucursal con longitud positiva en AR-B como dato corrupto
- Clave compuesta de productos: `(id_comercio, id_bandera, id_sucursal, id_producto)`